# **Domain specific small language model for African folktales**

**Team:** Team Ethiopie

**Organizing body:** TRI AI, short for Teaching, Research, Innovation in AI
for Africa, in partnership with AI Saturdays Lagos. This project is the
final stage team submission for Cohort 10 of TRI AI Saturdays, a 16 week
programme built around Google DeepMind's AI Research Foundations
curriculum, developed by Google DeepMind in collaboration with University
College London.

## Problem statement

The preservation of African folktales, and the cultural memory, values, and oral
traditions they carry, is a growing challenge. Fewer people pass these stories
down, and few digital tools exist to help communities preserve them. Existing
preservation efforts largely rely on manual transcription and scattered
anthropological archives, which are slow, incomplete, and rarely reach younger
generations. Current language models are trained mainly on Western text, so
they struggle to generate African stories that feel authentic to local dialects
and storytelling styles.

This project explores building a small, domain specific language model that
can generate narratives reflecting genuine regional traditions, dialects, and
storytelling structures. The goal is a practical tool for cultural
preservation, education, and creative storytelling, and a demonstration that
smaller, purpose built AI models can serve communities often left out of
mainstream AI development.

## Project overview

Given a short prompt describing a theme and a cultural region, our task is to
generate a folktale style story that resembles a hidden reference story
closely enough to score well under the competition's edit distance metric.
We approach this in stages, starting with a retrieval only baseline that
needs no training, then moving to supervised fine tuning and LoRA fine tuning
of a small pretrained instruct model.

## Scope of this notebook

The full vision above calls for training on real oral history recordings and
published literature across multiple African regions and languages. This
notebook is a smaller scale proof of concept. It works with a synthetic
folktale corpus provided as the competition benchmark. We use it to
demonstrate and validate the retrieval and fine tuning approaches at small
scale, as a step toward the larger goal of training on authentic, community
sourced material.

# **Literature review: Levenshtein distance and evaluation metrics**

### What is Levenshtein distance? An analogy

Imagine standing at a typewriter, trying to copy out a target sentence.
Every time you make a mistake, you have to make a manual correction.
Levenshtein distance simply counts how many corrections it takes to
turn your typed output into the exact target sentence.

It relies on three basic operations.

Insertion, adding a missing letter. For example, "cat" to "cats" is 1 edit.

Deletion, removing an unnecessary letter. For example, "tortoise" to
"tortois" is 1 edit.

Substitution, swapping one letter for another. For example, "Anansi"
to "Amansi" is 1 edit.

### A worked example

Here is how the algorithm measures the distance between a generated
word, "Kola," and a target reference word, "Kelo."

```
Generated:   K  o  l  a
             |  |  |  |
Operation:  keep sub keep sub
             |  |  |  |
Target:      K  e  l  o
```

Step by step:

1. Keep K, cost 0
2. Substitute o for e, cost plus 1
3. Keep l, cost 0
4. Substitute a for o, cost plus 1

Total Levenshtein distance equals 2.

### Mathematical definition

For two strings A and B, the minimum edit distance lev(A, B) is
defined recursively.

$$
\text{lev}(A, B) =
\begin{cases}
|A| & \text{if } |B| = 0 \\[4pt]
|B| & \text{if } |A| = 0 \\[4pt]
\text{lev}(\text{tail}(A), \text{tail}(B)) & \text{if } A[0] = B[0] \\[4pt]
1 + \min \big( \text{lev}(\text{tail}(A), B),\ \text{lev}(A, \text{tail}(B)),\ \text{lev}(\text{tail}(A), \text{tail}(B)) \big) & \text{otherwise}
\end{cases}
$$

Read in plain terms: if one string is empty, the distance is just the
length of the other string, since every remaining character has to be
inserted or deleted. If the first characters already match, we simply
move on to comparing the rest of the strings. Otherwise, we take the
cheapest of three choices, deleting a character, inserting a
character, or substituting one, and add its cost.

An efficient way to compute this was formalized by Wagner and Fischer
(1974) as a dynamic programming problem, now called the string to
string correction problem. This is the version implemented in most
modern edit distance libraries, and it runs in time proportional to m
times n, the product of the two string lengths, so it stays cheap for
short strings like ours.

### Why this matters for our folktale SLM

Because our competition scores generated stories strictly on character
level similarity against hidden reference stories, this metric
directly shapes our fine tuning strategy.

```
[ Input prompt: region and theme ]
              |
              v
[ Small language model ] ---> generated narrative
              |
              v  (Levenshtein distance check)
[ Hidden reference story ] ---> competition score
```

| Metric property | The reality | Practical strategy for our notebook |
| :--- | :--- | :--- |
| Paraphrase blindness | Penalizes synonym use. Saying "the lion roared loudly" instead of "the lion shouted loudly" loses points despite meaning the same thing. | Fine tune the SLM, for example via LoRA, directly on the target corpus's specific vocabulary and phrasing, rather than allowing open creative drift. |
| Character sensitivity | Small character errors, for example in regional names, add distance penalties quickly. | Our starter notebook already generates with `do_sample=False`, meaning greedy decoding, always picking the single most likely next token. This already favors precise, low variance output, so no extra temperature tuning is needed for this goal. |
| Length penalty | Extra sentences inflate insertion cost heavily, since every added character counts. | Keep `MAX_NEW_TOKENS` aligned with the reference story lengths we measured in EDA, roughly 9 to 64 words, rather than letting the model run long. |

### Summary

Levenshtein distance is a simple, well established way to measure how
different two strings are, counting the minimum number of single
character edits needed to turn one into the other. It has a long
history, from 1960s coding theory research to present day NLP
evaluation metrics like TER. For our project, the practical takeaway
is direct. Since our submissions are scored on closeness to a hidden
reference story at the character level, staying close to the source
corpus's specific wording and structure matters as much as, or more
than, getting the theme and moral of the story right.

### References

Devatine, N., and Abraham, L. (2024). Assessing human editing effort
on LLM generated texts via compression based edit distance. arXiv
preprint. https://arxiv.org/abs/2412.17321

Levenshtein, V. I. (1966). Binary codes capable of correcting
deletions, insertions, and reversals. Soviet Physics Doklady, 10(8),
707 to 710.

Snover, M., Dorr, B., Schwartz, R., Micciulla, L., and Makhoul, J.
(2006). A study of translation edit rate with targeted human
annotation. Proceedings of the 7th Conference of the Association for
Machine Translation in the Americas, 223 to 231.
https://aclanthology.org/2006.amta-papers.25/

Wagner, R. A., and Fischer, M. J. (1974). The string to string
correction problem. Journal of the ACM, 21(1), 168 to 173.
https://doi.org/10.1145/321796.321811
```


# **GPU setup and data loading**

In [1]:
# ==============================================================================
# SECTION 1: Environment Setup and Initialization
# Description:
# Sets up GPU environment variables and verifies single-device visibility 
# for memory-safe model execution on Kaggle GPU instances.
# ==============================================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
# ==============================================================================
# SECTION 2: Dataset Path Resolution and Data Loading
# Description:
# Automatically detects the runtime environment (Kaggle vs. local) and locates
# the competition dataset directory to load document sources and prompt tables.
# ==============================================================================

from pathlib import Path #handles file location in a way that works on any operating system

import pandas as pd #loads and work CSV files as tables

# This is the name Kaggle uses to identify our competition
COMPETITION_SLUG = "african-folktales-slm"

# Kaggle notebooks store data under /kaggle/input. If that folder exists,
# we know we are running on Kaggle. If not, we are running somewhere else,
# like a local computer, so we adjust how we look for our files.
ON_KAGGLE = Path("/kaggle/input").exists()


def find_data_dir(slug: str) -> Path:
    """
    This function's only job is to find the folder that contains our
    train_prompts.csv file, no matter how Kaggle happens to have named
    that folder this time. Kaggle does not always use the exact same
    folder name, so we check a few likely options before giving up.
    """

    # If we are not on Kaggle, our files are just sitting in the current folder
    if not ON_KAGGLE:
        return Path(".")

    # On Kaggle, all attached data lives somewhere under this root folder
    root = Path("/kaggle/input")

    # We try a few common folder name patterns, one at a time
    for pattern in (
        f"competitions/african-folktales-slm",
        f"competitions/{slug.replace('-', '_')}",
        slug,
    ):
        candidate = root / pattern
        # If train_prompts.csv exists inside this candidate folder, we found it
        if (candidate / "train_prompts.csv").exists():
            return candidate

    # If none of those patterns worked, we search every folder under root
    # until we find one that actually contains train_prompts.csv
    for path in root.rglob("train_prompts.csv"):
        return path.parent

    # If we still found nothing, something is wrong with how the data was
    # attached, so we stop and tell the user clearly what went wrong
    raise FileNotFoundError(
        "Could not find train_prompts.csv. Join the competition and attach its dataset."
    )


# Now we actually use the function above to find where our data lives
DATA_DIR = find_data_dir(COMPETITION_SLUG)

# We also decide where we are allowed to save new files we create.
# On Kaggle, only /kaggle/working is writable. Locally, the current folder works fine.
OUTPUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path(".")

# This line creates that output folder if it does not already exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Finally, we load each CSV file into a table (called a DataFrame in pandas)
# so we can start exploring and working with the data

docs = pd.read_csv(DATA_DIR / "documents.csv")               # the 24 source folktales
train = pd.read_csv(DATA_DIR / "train_prompts.csv")          # prompt to story pairs we learn from
test = pd.read_csv(DATA_DIR / "test_prompts.csv")            # prompts we must generate answers for
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")     # shows the required output format
baseline_submission = pd.read_csv(DATA_DIR / "baseline_submission.csv") # a worked example submission

# **Check for documents.CSV**

In [3]:
print("documents.csv, our 24 source folktales")
docs.head(24)

documents.csv, our 24 source folktales


,document_id,title,theme,culture_region,text,origin,source_url,license
0,doc_tri_001,The spider and the shared pot,trickster,west_africa,"When famine visited the village, the spider tr...",synthetic,NaN,CC0-1.0
1,doc_tri_002,Tortoise and the river festival,trickster,southern_africa,Tortoise could not swim but wished to attend t...,synthetic,NaN,CC0-1.0
2,doc_tri_003,Hare and the drum of thunder,trickster,east_africa,Hare found a hollow log that echoed like thund...,synthetic,NaN,CC0-1.0
3,doc_ori_001,Why the baobab looks upside down,origin_myth,east_africa,The first baobab boasted that its roots could ...,synthetic,NaN,CC0-1.0
4,doc_ori_002,How the river learned to bend,origin_myth,central_africa,Long ago the river ran straight and forgot the...,synthetic,NaN,CC0-1.0
5,doc_ori_003,The first millet song,origin_myth,west_africa,Millet seeds were shy and hid beneath stones u...,synthetic,NaN,CC0-1.0
6,doc_mor_001,The girl who returned the lost cowrie,moral_tale,west_africa,A merchant dropped a cowrie shell in the marke...,synthetic,NaN,CC0-1.0
7,doc_mor_002,Two brothers and one path,moral_tale,southern_africa,Two brothers inherited one path to the grazing...,synthetic,NaN,CC0-1.0
8,doc_mor_003,The pot that would not boil for greed,moral_tale,east_africa,A traveler asked for one ladle of stew and was...,synthetic,NaN,CC0-1.0
9,doc_her_001,Daughter of the copper hills,hero_journey,central_africa,She crossed three valleys to fetch medicine ba...,synthetic,NaN,CC0-1.0


In [4]:
print("Shape of documents.csv")
print("Rows:", docs.shape[0], "| Columns:", docs.shape[1])
print()
print("Column names:", list(docs.columns))

Shape of documents.csv
Rows: 24 | Columns: 8

Column names: ['document_id', 'title', 'theme', 'culture_region', 'text', 'origin', 'source_url', 'license']


In [5]:
# unique() lists every distinct value found in a column, with no repeats.
# This tells us exactly which themes and regions exist in our corpus.

print("Distinct themes in documents.csv:")
print(docs["theme"].unique())
print()
print("Distinct culture regions in documents.csv:")
print(docs["culture_region"].unique())

Distinct themes in documents.csv:
['trickster' 'origin_myth' 'moral_tale' 'hero_journey' 'animal_fable'
 'community_wisdom']

Distinct culture regions in documents.csv:
['west_africa' 'southern_africa' 'east_africa' 'central_africa' 'diaspora']


In [6]:
# value_counts() tells us how many rows fall into each category.
# This shows us whether our corpus is balanced across themes and regions,
# or whether some categories have far more examples than others.

print("How many documents per theme:")
print(docs["theme"].value_counts())
print()
print("How many documents per region:")
print(docs["culture_region"].value_counts())

How many documents per theme:
theme
trickster           4
origin_myth         4
moral_tale          4
hero_journey        4
animal_fable        4
community_wisdom    4
Name: count, dtype: int64

How many documents per region:
culture_region
west_africa        6
east_africa        6
southern_africa    5
central_africa     5
diaspora           2
Name: count, dtype: int64


In [7]:
# isna().sum() counts how many empty (missing) values exist in each column

print("Missing values per column in documents.csv")
docs.isna().sum()

Missing values per column in documents.csv


document_id        0
title              0
theme              0
culture_region     0
text               0
origin             0
source_url        24
license            0
dtype: int64

In [8]:
docs = docs.drop(columns=['source_url'])


In [9]:
# str.split().str.len() counts the number of words in each story's text.
# describe() then gives us the average, shortest, longest, and spread.

docs["word_count"] = docs["text"].str.split().str.len()
print("Word count summary for documents.csv text")
docs["word_count"].describe()

Word count summary for documents.csv text


count    24.000000
mean     48.333333
std       6.624504
min      37.000000
25%      43.750000
50%      47.500000
75%      52.000000
max      64.000000
Name: word_count, dtype: float64

# **Check for train_prompts.CSV**

In [10]:
train.head()

,prompt,theme,culture_region,document_id,reference_story,PromptId
0,Explain in story form why the baobab looks ups...,origin_myth,east_africa,doc_ori_001,The first baobab boasted that its roots could ...,1
1,Tell a story showing generosity warms a commun...,moral_tale,east_africa,doc_mor_003,A traveler asked for one ladle of stew and was...,2
2,Tell an origin myth about a river learning to ...,origin_myth,central_africa,doc_ori_002,Long ago the river ran straight and forgot the...,3
3,Medicine bark across three valleys — hero tale.,hero_journey,central_africa,doc_her_001,She braided grass ropes with strangers in a st...,4
4,Write a fable where elephant learns from firef...,animal_fable,central_africa,doc_ani_002,Elephant demanded the forest paths be cleared ...,5


In [11]:
print("Shape of train_prompts.csv")
print("Rows:", train.shape[0], "| Columns:", train.shape[1])
print()
print("Column names:", list(train.columns))

Shape of train_prompts.csv
Rows: 38 | Columns: 6

Column names: ['prompt', 'theme', 'culture_region', 'document_id', 'reference_story', 'PromptId']


In [12]:
print("Distinct themes in train_prompts.csv:")
print(train["theme"].unique())
print()
print("Distinct culture regions in train_prompts.csv:")
print(train["culture_region"].unique())

Distinct themes in train_prompts.csv:
['origin_myth' 'moral_tale' 'hero_journey' 'animal_fable'
 'community_wisdom' 'trickster']

Distinct culture regions in train_prompts.csv:
['east_africa' 'central_africa' 'southern_africa' 'west_africa' 'diaspora']


In [13]:
print("How many training prompts per theme:")
print(train["theme"].value_counts())
print()
print("How many training prompts per region:")
print(train["culture_region"].value_counts())

How many training prompts per theme:
theme
community_wisdom    7
animal_fable        7
moral_tale          6
origin_myth         6
hero_journey        6
trickster           6
Name: count, dtype: int64

How many training prompts per region:
culture_region
west_africa        10
east_africa         8
central_africa      8
southern_africa     8
diaspora            4
Name: count, dtype: int64


In [14]:
train["word_count"] = train["reference_story"].str.split().str.len()
print("Word count summary for reference_story in train_prompts.csv")
train["word_count"].describe()

Word count summary for reference_story in train_prompts.csv


count    38.000000
mean     30.078947
std      17.015703
min       9.000000
25%      15.000000
50%      21.000000
75%      44.750000
max      64.000000
Name: word_count, dtype: float64

In [15]:
print("Missing values per column in train_prompts.csv")
train.isna().sum()

Missing values per column in train_prompts.csv


prompt             0
theme              0
culture_region     0
document_id        0
reference_story    0
PromptId           0
word_count         0
dtype: int64

In [16]:
# This checks whether every document in documents.csv actually gets used
# by at least one training prompt, or if some documents are left unused

used_docs = train["document_id"].nunique()
total_docs = docs["document_id"].nunique()
print(f"{used_docs} out of {total_docs} documents are referenced in train_prompts.csv")

24 out of 24 documents are referenced in train_prompts.csv


# **Check for 'test_prompts.CSV'**

In [17]:
test.head()


,PromptId,prompt,theme,culture_region
0,1001,Tell a moral market tale about returning a los...,moral_tale,west_africa
1,1002,Create a hero story of an orphan who saves fis...,hero_journey,east_africa
2,1003,Create an East African tale of hare claiming t...,trickster,east_africa
3,1004,Hyena jumps at the moon in water.,animal_fable,east_africa
4,1005,Tell a trickster tale of monkey guarding a hon...,trickster,central_africa


In [18]:
print("Shape of test_prompts.csv")
print("Rows:", test.shape[0], "| Columns:", test.shape[1])
print()
print("Column names:", list(test.columns))

Shape of test_prompts.csv
Rows: 10 | Columns: 4

Column names: ['PromptId', 'prompt', 'theme', 'culture_region']


In [19]:
print("Distinct themes in test_prompts.csv:")
print(test["theme"].unique())
print()
print("Distinct culture regions in test_prompts.csv:")
print(test["culture_region"].unique())

Distinct themes in test_prompts.csv:
['moral_tale' 'hero_journey' 'trickster' 'animal_fable' 'origin_myth'
 'community_wisdom']

Distinct culture regions in test_prompts.csv:
['west_africa' 'east_africa' 'central_africa' 'southern_africa']


In [20]:
print("How many test prompts per theme:")
print(test["theme"].value_counts())
print()
print("How many test prompts per region:")
print(test["culture_region"].value_counts())

How many test prompts per theme:
theme
moral_tale          2
hero_journey        2
trickster           2
origin_myth         2
animal_fable        1
community_wisdom    1
Name: count, dtype: int64

How many test prompts per region:
culture_region
east_africa        4
west_africa        2
central_africa     2
southern_africa    2
Name: count, dtype: int64


In [21]:
print("Missing values per column in test_prompts.csv")
test.isna().sum()

Missing values per column in test_prompts.csv


PromptId          0
prompt            0
theme             0
culture_region    0
dtype: int64

In [22]:
# ==============================================================================
# SECTION 4: TF-IDF Retrieval Baseline
# Description:
# Generates a non-parametric retrieval baseline using TF-IDF n-gram vectorization
# and cosine similarity to match test prompts with reference training stories.
# ==============================================================================

# Import TF-IDF Vectorizer to convert text prompts into numerical feature matrices
from sklearn.feature_extraction.text import TfidfVectorizer

# Import cosine_similarity to compute similarity scores between vector representations
from sklearn.metrics.pairwise import cosine_similarity

# Initialize an empty list to store the predicted PromptId and Story pairs for test data
rows = []

# Loop through each individual row in the test prompts DataFrame
for _, t in test.iterrows():
    # Filter the training dataset to only include rows with the same theme as the test prompt
    sub = train[train["theme"] == t["theme"]]
    # Fallback check: If no matching theme exists in training data, use the full training set
    if sub.empty:
        sub = train

    # Initialize a TF-IDF Vectorizer using unigrams and bigrams (1-word and 2-word sequences)
    vec = TfidfVectorizer(ngram_range=(1, 2))

    # Convert all training prompt texts in the filtered subset into TF-IDF numerical matrices
    sub_vectors = vec.fit_transform(sub["prompt"])

    # Convert the single current test prompt text into the same TF-IDF matrix space
    test_vector = vec.transform([t["prompt"]])

    # Calculate cosine similarity scores between the test prompt vector and all training vectors
    sims = cosine_similarity(test_vector, sub_vectors)

    # Identify the index with the highest similarity score and retrieve its corresponding reference story
    ans = sub.iloc[sims.argmax()]["reference_story"]

    # Append the test PromptId and the matched story output to our result tracking list
    rows.append({"PromptId": t["PromptId"], "Story": ans})

baseline_submission = pd.DataFrame(rows)
baseline_submission.to_csv(OUTPUT_DIR / "submission.csv", index=False)

print("Wrote", OUTPUT_DIR / "submission.csv")
baseline_submission.head()

Wrote /kaggle/working/submission.csv


,PromptId,Story
0,1001,A boy blamed goats for a broken gourd he had d...
1,1002,Practice by moonlight let her cast the chief's...
2,1003,Hare borrowed thunder from a hollow log until ...
3,1004,Hyena saw the moon in a still pond and leapt t...
4,1005,Bees marked the hive with a song only honest h...


## Build SFT/LoRA training examples

Turn every `train_prompts.csv` row into a `prompt + story` training example, using the matching source document as a style cue.

In [23]:
# ==============================================================================
# SECTION 5: Prompt Formatting Engine & Dataset Construction
# Description:
# Formats instruction prompts paired with reference target stories for supervised
# fine-tuning (SFT) and LoRA adapter training.
# ==============================================================================

# Define a function to construct Gemma-2 structured chat prompt input strings
def build_prompt_input(theme, region, prompt, style_cue):
    return (
        # Start user turn tag in Gemma template format
        f"<start_of_turn>user\n"
        # Format metadata including storytelling theme and cultural region origin
        f"Theme: {theme} | Region: {region}\n"
        # Add the target instruction or user generation prompt
        f"Prompt: {prompt}\n"
        # Append style text context truncated to 250 characters to prevent prompt bloat
        f"Style cue: {style_cue[:250]}<end_of_turn>\n"
        # Open model turn tag indicating where generation should begin
        f"<start_of_turn>model\n"
    )

# Initialize an empty list to hold structured training dictionaries
sft_data = []

# Iterate over every sample in the training prompts DataFrame
for _, r in train.iterrows():
    # Retrieve matching document text from docs DataFrame using document_id key
    doc_text = docs.loc[docs.document_id == r.document_id, "text"].iloc[0]
    
    # Generate the formatted input prompt string using metadata and document style
    prompt_str = build_prompt_input(r.theme, r.culture_region, r.prompt, doc_text)
    
    # Concatenate the prompt with the reference story and model end token tag
    full_text = prompt_str + r.reference_story + "<end_of_turn>"
    
    # Append structured dictionary containing both prompt and target full text
    sft_data.append({"prompt": prompt_str, "full_text": full_text})

In [24]:
# Quick check to preview the first formatted training sample
print(f"Total samples processed: {len(sft_data)}")
print("\n--- SAMPLE PROMPT ---")
print(sft_data[0]["prompt"])
print("\n--- SAMPLE FULL TEXT ---")
print(sft_data[0]["full_text"])

Total samples processed: 38

--- SAMPLE PROMPT ---
<start_of_turn>user
Theme: origin_myth | Region: east_africa
Prompt: Explain in story form why the baobab looks upside down.
Style cue: The first baobab boasted that its roots could drink any star. The soil spirit grew tired of pride and planted the tree head-down so its branches learned humility underground. When travelers rest beneath its wide trunk, they remember: greatness must b<end_of_turn>
<start_of_turn>model


--- SAMPLE FULL TEXT ---
<start_of_turn>user
Theme: origin_myth | Region: east_africa
Prompt: Explain in story form why the baobab looks upside down.
Style cue: The first baobab boasted that its roots could drink any star. The soil spirit grew tired of pride and planted the tree head-down so its branches learned humility underground. When travelers rest beneath its wide trunk, they remember: greatness must b<end_of_turn>
<start_of_turn>model
The first baobab boasted that its roots could drink any star. The soil spirit gr

In [25]:
# ==============================================================================
# SECTION 6: Model Loading and Hardware Compatibility Patches
# Description:
# Configures path variables, checks file existence on Kaggle, and applies 
# stability patches for LoRA setup.
# ==============================================================================

# Import PEFT utility functions to inspect and handle library dependencies
import peft.import_utils

# Import PEFT torchao sub-module to manage lower-level quantization bindings
import peft.tuners.lora.torchao

# Override torchao availability checks to avoid Kaggle GPU initialization errors
peft.import_utils.is_torchao_available = lambda: False
peft.tuners.lora.torchao.is_torchao_available = lambda: False

# Define local input directory path pointing to pre-downloaded Gemma-2-2B-IT model weights
MODEL_PATH = "/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2"

# Print the model path being targeted to confirm variable assignment
print("Using:", MODEL_PATH)

# Import os module to interact with the file system
import os

# List and display the contents of the model directory to verify configuration files exist
print(os.listdir(MODEL_PATH))


Using: /kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2
['model.safetensors.index.json', 'config.json', 'model-00001-of-00002.safetensors', 'model-00002-of-00002.safetensors', 'README.md', 'tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json', '.gitattributes', 'tokenizer.model', 'generation_config.json']


In [26]:
# Locate Gemma config file dynamically
def locate_gemma():
    input_path = Path("/kaggle/input")
    for p in input_path.rglob("config.json"):
        if "gemma" in str(p).lower():
            return p.parent
    return None

detected_path = locate_gemma()
if detected_path:
    MODEL_PATH = str(detected_path)
    print("Successfully attached! Updated MODEL_PATH:", MODEL_PATH)
    print("Directory contents:", os.listdir(MODEL_PATH))
else:
    print("Still not detected. Double check if the dataset/model finished mounting in the right panel.")

Successfully attached! Updated MODEL_PATH: /kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2
Directory contents: ['model.safetensors.index.json', 'config.json', 'model-00001-of-00002.safetensors', 'model-00002-of-00002.safetensors', 'README.md', 'tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json', '.gitattributes', 'tokenizer.model', 'generation_config.json']


In [27]:
import peft.import_utils
import peft.tuners.lora.torchao

peft.import_utils.is_torchao_available = lambda: False
peft.tuners.lora.torchao.is_torchao_available = lambda: False
print("Patched torchao check")


Patched torchao check


## LoRA fine-tune (4-bit, single GPU, memory-safe)

Loads the model in 4-bit to fit Kaggle's GPU memory, trains LoRA adapters on the 38 examples, generates stories for the 10 test prompts, and overwrites `submission.csv`.

`USE_LORA` is off by default so this cell doesn't accidentally burn GPU time — set it to `True` once everything above has run cleanly.

Before executing the code in this section, it helps to understand the underlying strategy behind fine-tuning small language models. When training a model like Gemma-2 on a specialized dataset, there is no single "correct" way to learn; instead, engineering choices dictate how effectively a model absorbs new patterns.

Below is a short technical narrative comparing two distinct technical approaches to parameter-efficient fine-tuning (LoRA).

### The Fine-Tuning Narrative

Imagine teaching a student to write short African folktales based on a structured prompt. You hand them a workbook containing 38 examples. Each page has a **Header/Prompt** (*Theme, Region, Style Cue*) followed by the **Target Story**.

How you instruct the student to study determines their final performance:

#### 🛣️ Pathway A: The Standard Baseline Approach

In the standard baseline approach, the student is instructed to memorize **everything** on the page equally—both the prompt instructions and the story answer. Every time they read a prompt header like *"Theme: Wisdom"*, they spend mental effort predicting those exact words.

* **The Mechanism:** Standard Causal Language Modeling calculates cross-entropy loss over every single token in a sequence.
* **The Limitation:**
* **Low Capacity ($r=8$):** The LoRA Rank ($r$) defines the inner dimension of the trainable adapter matrices added to the model. An $r=8$ setting gives the student a narrow "notebook" with limited parameter space to store new knowledge.
* **Attention-Only Modules (`q_proj`, `v_proj`):** Restricting updates to query and value attention projections only allows the model to adjust *where* it looks in a sequence, not *what* words or cultural phrasing it chooses to output.


* **The Result:** The student spends half their energy memorizing static instruction headers instead of story writing. Unable to capture deep narrative style, training halts at a high residual loss of **~2.19**.

---

#### 🚀 Pathway B: The Masked & Expanded Approach (Our Strategy)

In our optimized approach, we place a physical mask over the prompt headers using PyTorch's **`-100` label masking index**. In PyTorch, `-100` acts as an `ignore_index` signal: whenever the loss function hits a token labeled `-100`, it assigns it $0.0$ loss and completely skips backpropagation for it. We tell the student: *"Ignore the prompt words entirely during grading; focus 100% of your effort strictly on generating the target story text."*

* **The Architectural Shift:**
* **Prompt Loss Masking (`-100` Indexing):** By injecting `-100` across all prompt tokens, $100\%$ of gradient updates focus strictly on predicting story tokens.
* **Doubled Capacity ($r=16$ & $\alpha=32$):** Doubling rank $r$ from $8$ to $16$ expands the adapter's mathematical bottleneck, doubling its capacity to learn rich sentence structures and narrative tone. Scaling $\alpha$ to $32$ maintains a stable $2.0$ update ratio ($\frac{\alpha}{r}$).
* **Full MLP Layer Targeting:** In addition to attention layers, we target feed-forward MLP projections (`gate_proj`, `up_proj`, `down_proj`). While attention controls context focus, MLP layers process factual recall and vocabulary selection—giving the model direct control over writing style.


* **The Result:** Combining prompt masking, expanded adapter rank, extended context window ($512$ tokens), and a smooth cosine learning rate decay ensures every single weight update improves output quality—driving training loss down to **~0.0073**.

---


---

> ⚠️ **Execution Note:** Section 4c runs in 4-bit quantization to fit safely inside Kaggle's single GPU memory limit. The `USE_LORA` flag is set to `False` by default to prevent burning GPU quota. Once you are ready, toggle `USE_LORA = True` in the code cell below to start training.
> 
>

# USING A STANDARD BASLINE SFT

## Final validation before submitting



In [28]:
final = pd.read_csv(OUTPUT_DIR / "submission.csv")
assert list(final.columns) == ["PromptId", "Story"]
assert len(final) == len(test)
assert final["PromptId"].tolist() == test["PromptId"].tolist()
assert final["PromptId"].is_unique
assert final["Story"].notna().all()
assert (final["Story"].str.len() > 0).all()
print("submission.csv looks valid:", final.shape)
final.head(10)


submission.csv looks valid: (10, 2)


,PromptId,Story
0,1001,A boy blamed goats for a broken gourd he had d...
1,1002,Practice by moonlight let her cast the chief's...
2,1003,Hare borrowed thunder from a hollow log until ...
3,1004,Hyena saw the moon in a still pond and leapt t...
4,1005,Bees marked the hive with a song only honest h...
5,1006,Long ago the river ran straight and forgot the...
6,1007,Long ago the river ran straight and forgot the...
7,1008,Two brothers inherited one path to the grazing...
8,1009,Gratitude on the baobab drum brought gentle ra...
9,1010,Villagers argued whose son would lead the harv...


# USING AN OPTIMIZED LOSS- MASKED LORA

In [29]:
# ==============================================================================
# SECTION 7: Supervised Fine-Tuning (SFT) with Loss Masking Optimization
# Description:
# Configures LoRA adapters, sets prompt-token loss masking (-100), and trains
# using dynamic seq2seq collators to directly minimize causal loss.
# ==============================================================================

# Control flag to execute LoRA training routine
USE_LORA = True

# Define maximum context length to avoid truncating stories mid-sentence
MAX_SEQ_LENGTH = 512

# Define maximum target token generation length during evaluation inference
MAX_NEW_TOKENS = 180

if USE_LORA:
    # Import core deep learning frameworks and SFT training tools
    import torch
    from datasets import Dataset
    from peft import LoraConfig, TaskType, get_peft_model
    from transformers import (
        AutoModelForCausalLM, 
        AutoTokenizer, 
        Trainer, 
        TrainingArguments, 
        DataCollatorForSeq2Seq
    )

    # Load tokenizer instance matching local Gemma weights directory
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=ON_KAGGLE)
    
    # Assign end-of-sequence token as padding token if pad_token is missing
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Load base Causal LM in half precision (float16) mapped directly to GPU device 0
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.float16,
        device_map={"": 0},
        local_files_only=ON_KAGGLE,
    )

    # Configure Low-Rank Adaptation parameters across attention and projection layers
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        task_type=TaskType.CAUSAL_LM,
    )
    
    # Attach trainable LoRA adapter parameters onto base frozen model layers
    model = get_peft_model(base_model, peft_config)
    
    # Print summary of trainable vs frozen weight count
    model.print_trainable_parameters()

    # Preprocessing function that applies prompt label masking (-100) for cross-entropy loss
    def tokenize_and_mask(examples):
        input_ids_list, labels_list, attention_mask_list = [], [], []
        
        for p_text, f_text in zip(examples["prompt"], examples["full_text"]):
            # Tokenize isolated prompt text and full target sequence separately
            p_tokens = tokenizer(p_text, add_special_tokens=False)["input_ids"]
            f_tokens = tokenizer(f_text, add_special_tokens=False, max_length=MAX_SEQ_LENGTH, truncation=True)["input_ids"]
            
            # Construct label array replacing prompt indices with -100 to ignore prompt loss
            labels = [-100] * len(p_tokens) + f_tokens[len(p_tokens):]
            
            input_ids_list.append(f_tokens)
            labels_list.append(labels[:len(f_tokens)])
            attention_mask_list.append([1] * len(f_tokens))

        return {
            "input_ids": input_ids_list,
            "labels": labels_list,
            "attention_mask": attention_mask_list
        }

    # Convert raw memory list into Hugging Face Dataset format
    dataset = Dataset.from_dict({
        "prompt": [x["prompt"] for x in sft_data],
        "full_text": [x["full_text"] for x in sft_data]
    })
    
    # Apply tokenization and loss masking across dataset splits
    tokenized_ds = dataset.map(tokenize_and_mask, batched=True, remove_columns=["prompt", "full_text"])

    # Configure training hyperparameters including learning rate schedule and accumulation
    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR / "lora-checkpoints"),
        num_train_epochs=5,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=3e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        logging_steps=2,
        save_strategy="no",
        fp16=torch.cuda.is_available(),
        report_to="none",
    )

    # Initialize Hugging Face Trainer with dynamic sequence batch collator
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, pad_to_multiple_of=8, return_tensors="pt"),
    )
    
    # Execute fine-tuning loop
    trainer.train()

    # Generate fine-tuned model inferences for test prompts
    results = []
    for _, t in test.iterrows():
        # Match test prompt theme with available train sample documents
        sub = train[train["theme"] == t.theme]
        row = sub.iloc[0] if len(sub) else train.iloc[0]
        doc = docs.loc[docs.document_id == row.document_id, "text"].iloc[0]
        
        # Build evaluation prompt and move input tensors to GPU
        prompt_text = build_prompt_input(t.theme, t.culture_region, t.prompt, doc)
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        
        # Generate model story tokens without computing gradient graphs
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
            
        # Decode output tokens into text, stripping out input prompt prefix
        decoded = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        results.append({"PromptId": t["PromptId"], "Story": decoded.strip()})

    # Save outputs to target submission CSV file
    submission = pd.DataFrame(results)
    submission.to_csv(OUTPUT_DIR / "submission.csv", index=False)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

trainable params: 20,766,720 || all params: 2,635,108,608 || trainable%: 0.7881


Map:   0%|          | 0/38 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
2,1.948522
4,1.893407
6,2.377910
8,1.224562
10,0.845624
12,0.645284
14,0.603406
16,0.366449
18,0.973032
20,0.815499


In [30]:
# ==============================================================================
# SECTION 8: Submission Integrity and Format Validation
# Description:
# Performs schema validation, null checks, and row count verification on the 
# generated submission file before final submission.
# ==============================================================================

# Read the newly generated submission CSV back into a pandas DataFrame
final = pd.read_csv(OUTPUT_DIR / "submission.csv")

# Verify that column headers exactly match competition requirements
assert list(final.columns) == ["PromptId", "Story"], "Column names do not match expected schema!"

# Ensure total row count matches the test set row count exactly
assert len(final) == len(test), f"Expected {len(test)} rows, but got {len(final)}!"

# Confirm prompt IDs align perfectly with test set ordering
assert final["PromptId"].tolist() == test["PromptId"].tolist(), "Prompt ID sequence mismatch!"

# Verify that no predictions contain NaN or null values
assert final["Story"].notna().all(), "Submission contains missing (NaN) values!"

# Ensure that no generated story is an empty string
assert (final["Story"].str.len() > 0).all(), "Submission contains empty string outputs!"

# Print confirmation message and output dimensions if all assertions pass
print("Submission integrity verified successfully:", final.shape)

# Preview the top generated predictions
final.head(10)

Submission integrity verified successfully: (10, 2)


,PromptId,Story
0,1001,A traveler asked for one ladle of stew and was...
1,1002,She crossed three valleys to fetch medicine ba...
2,1003,Bees marked the hive with a song only honest h...
3,1004,Hyena jumped at the moon in water.
4,1005,Bees marked the hive with a song only honest h...
5,1006,The first baobab boasted that its roots could ...
6,1007,Grandmother sings and the river bends — myth.
7,1008,A traveler asked for one ladle of stew and was...
8,1009,She crossed three valleys to fetch medicine ba...
9,1010,Grandmother's circle fixed the fence by sunset...
